In [ ]:
# ! pip install langchain langchain-openai langchain-community langgraph python-dotenv faiss-cpu pypdf




In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

In [4]:

from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="qwen3:4b",
    temperature=0,
    think=False
)


In [5]:
load_dotenv()

True

In [6]:
loader = PyPDFLoader('islr.pdf')
docs = loader.load()

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/Encoding': '/MacRomanEncoding', '/FontDescriptor': IndirectObject(1584, 0, 2177188821744), '/BaseFont': '/Times-Roman', '/Widths': [250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 333, 408, 500, 500, 833, 778, 180, 333, 333, 500, 564, 250, 333, 250, 278, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 278, 278, 564, 564, 564, 444, 921, 722, 667, 667, 722, 611, 556, 722, 722, 333, 389, 722, 611, 889, 722, 722, 556, 722, 667, 556, 611, 722, 722, 944, 722, 722, 611, 333, 278, 333, 469, 500, 333, 444, 500, 444, 500, 444, 333, 500, 500, 278, 278, 500, 278, 778, 500, 500, 500, 500, 333, 389, 278, 500, 500, 722, 500, 500, 444, 480, 200, 480, 541, 250, 722, 722, 667, 611, 722, 722, 722, 444, 444, 444, 444, 444, 444, 444, 444, 444, 444, 444, 278, 2

In [7]:
len(docs)

441

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [9]:
len(chunks)

1308

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint   
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = FAISS.from_documents(chunks, emb)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# vs

In [ ]:
retriever = vs.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [ ]:
@tool

def rag_tool(query):

    """
    Retrieve relevant information from the pdf document.
    Use this tool when user asks factual /conceptual questions
    that minght be answerd from the stored dcument   
    """

    result = retriever.invoke(query)

    context = [docs.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query': query,
        'context':context,
        'metadata':metadata
    }

In [ ]:
tools = [rag_tool]
llm_with_tool = llm.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
    messages :Annotated[list[BaseMessage],add_messages]

In [ ]:
def chatnode(state:ChatState):
    message = state['messages']

    response = llm_with_tool.invoke(message)

    return {'messages':response}

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(ChatState)

graph.add_node('chat_node',chatnode)
graph.add_node('tool_node',tool_node)

graph.add_edge(START,'chatnode')
graph.add_conditional_edges('chatnode',tools_condition)
graph.add_edge('tool_node','chatnode')


chatbot = graph.compile()


In [ ]:
chatbot

In [ ]:
result = chatbot.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Using the pdf notes, explain what is logistic regression"
                )
            )
        ]
    }
)

In [ ]:
print(result['messages'][-1].content)